# Hand-check: three well-known recalls

Week 1 checkpoint from the project plan: pick one air bag, one engine fire and one brake recall, and confirm
that their complaints show up **before** the recall date, and that our labels mark those cars.

| Type | Campaign | Recall date | Cars | Defect |
|---|---|---|---|---|
| Air bag | 18V-137 | 2018-02-27 | Hyundai Sonata / Sonata Hybrid 2011-13 | air bag control unit can fail, so air bags may not deploy (not Takata) |
| Engine fire | 19V-120 | 2019-02-22 | Kia Soul 1.6L 2012-16 | engine damage that can lead to a fire |
| Brake | 19V-645 | 2019-09-06 | GM full-size trucks and SUVs 2014-18 | brake vacuum pump wears, pedal goes hard |

Complaints are matched on make + model + model year + component in the cleaned `stg.complaints` / `stg.recalls`
tables (complaints 2010 to 2026), dated by `ldate`. One row per complaint (`odi_no`). `raw_name_share` is the share of
earlier complaints whose raw part name equals the recall's raw part name, which is what a join without the part map sees.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
con = duckdb.connect(str(ROOT / "data" / "nhtsa.duckdb"), read_only=True)
con.execute(f"create temp view labels as select * from read_parquet('{(ROOT / 'data/processed/recall_labels_stg.parquet').as_posix()}')")

RECALLS = ["18V137000", "19V120000", "19V645000"]

In [ ]:
def load(camp):
    # the recalled cars and part, then every complaint on those cars about the same cleaned part
    con.execute(f"""create or replace temp view cars as
        select distinct make, model, model_year, component, split_part(component_raw, ':', 1) as raw_part, recall_date
        from stg.recalls where camp_no = '{camp}'""")
    con.execute("""create or replace temp view cmp as
        select c.odi_no, min(c.ldate) as received, max(c.is_fire) as fire, max(c.is_crash) as crash,
               string_agg(distinct split_part(c.component_raw, ':', 1), ' | ') as part,
               -- would the raw part names have matched the recall's?
               max((split_part(c.component_raw, ':', 1) in (select raw_part from cars))::int) as raw_name_match,
               any_value(c.narrative) as narrative
        from stg.complaints c
        join (select distinct make, model, model_year, component from cars) k using (make, model, model_year, component)
        group by c.odi_no""")
    return con.execute("select min(recall_date) from cars").fetchone()[0]


def summary(camp):
    rc = load(camp)
    s = con.execute(f"""
        select count(*) filter (where received < date '{rc}') as before_recall,
               count(*) filter (where received >= date '{rc}') as after_recall,
               min(received) as first_complaint,
               sum(fire) filter (where received < date '{rc}') as fire_before,
               sum(crash) filter (where received < date '{rc}') as crash_before,
               round(avg(raw_name_match) filter (where received < date '{rc}'), 3) as raw_name_share
        from cmp""").df()
    s.insert(0, "recall_date", rc)
    s.insert(0, "campaign", camp)
    return s


def plot(camp, ax):
    rc = load(camp)
    m = con.execute(f"""select date_trunc('month', received) as month, count(*) as n from cmp
        where received >= date '{rc}' - interval 24 month and received < date '{rc}' + interval 12 month
        group by 1 order by 1""").df()
    ax.bar(m.month, m.n, width=25, color="#b86a00", alpha=0.6)
    ax.axvline(rc, color="black", lw=1.5)
    ax.set_title(f"{camp}  (recall {rc})", fontsize=10)
    ax.set_ylabel("complaints / month")


def labels_before(camp):
    rc = load(camp)
    return con.execute(f"""
        select l.component, l.next_recall_campaign_ids, count(*) cell_months, sum(y_6m::int) y_6m_true
        from labels l
        join (select distinct make, model, model_year, component from cars) k using (make, model, model_year, component)
        where l.month >= date '{rc}' - interval 6 month and l.month < date_trunc('month', date '{rc}')
        group by all order by 1, 3 desc""").df()


def narratives(camp, n=3):
    rc = load(camp)
    return con.execute(f"""select odi_no, received, part, left(narrative, 300) as text from cmp
        where received < date '{rc}' order by received desc limit {n}""").df()

## 1. Do complaints come before the recall?

In [ ]:
pd.concat([summary(c) for c in RECALLS], ignore_index=True)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=False)
for ax, c in zip(axes, RECALLS):
    plot(c, ax)
fig.suptitle("Complaints on the recalled part, 24 months before to 12 months after (black line = recall)")
fig.tight_layout()

## 2. What did people actually write just before the recall?

In [ ]:
for c in RECALLS:
    print("==", c)
    for r in narratives(c).itertuples():
        print(f"  {r.odi_no}  {r.received}  [{r.part}]  {r.text}")
    print()

## 3. Do our labels mark these cars in the 6 months before the recall?

In [ ]:
for c in RECALLS:
    print("==", c)
    print(labels_before(c).to_string(index=False))
    print()

## Findings

Run on the cleaned tables, Nov 16.

1. **All three pass the basic test.** Each recall had complaints about the same part on the same cars
   well before the recall date: 307 for the Sonata (first in Oct 2010, 51 with a crash), 107 for the Kia Soul
   engine (29 of them fires), 1,176 for the GM brakes (94 with a crash). The Sonata campaign lists air bags and
   seat belts, so its count covers both parts.
2. **The shape of the ramp differs.**
   - GM brakes: a clear ramp, from about 15 a month to 90 or more a month roughly ten months before the recall.
     This is the case the model should catch early.
   - Kia Soul: a small rise in the 9 months before the recall, then a big jump *after* it (2,298 complaints after
     vs. 107 before). The jump is why post-recall complaints must never count toward the next flag (leakage rule 3).
   - Sonata: a steady 5 to 20 complaints a month with no clear rise. It is hard to flag from counts alone;
     the text ("air bag light stays on") is the better signal.
3. **The part map fixed the GM case.** Only 2% of the earlier GM complaints use the same raw part name as the recall
   (`SERVICE BRAKES` vs. `SERVICE BRAKES, HYDRAULIC`). With the cleaned name `BRAKES` they all join, and all 110
   cell-months in the 6 months before the recall are labeled positive with campaign 19V645000. The Sonata and
   Kia Soul recalls already used the same part names as their complaints, and their cell-months are all labeled too.
4. **Some complaints mention earlier recalls.** These aren't flagged `RC` or `EWR`, so a "recall" keyword
   feature would need care.